<a href="https://colab.research.google.com/github/tallclub/matimo/blob/main/docs/notebooks/05_mcp_server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Matimo Notebook 05: MCP Server - Expose Matimo Tools Over the Model Context Protocol

The [Model Context Protocol (MCP)](https://modelcontextprotocol.io/) is how Claude Desktop, Cursor, Windsurf, and other MCP-compatible clients discover and call tools. Matimo can wrap any set of tools (built-in, provider packages, or your own YAML tools) and expose them as a standard MCP server - with governance (auth stripping, approval gating) already applied.

### What you'll learn
- Wrapping a `Matimo` instance in an `MCPServer` for **stdio** transport (the Claude Desktop / Cursor integration path)
- Connecting with the official `mcp` Python client, listing tools, and calling one
- Skills auto-exposed as MCP **resources** (`skills://<name>`) - no extra tool call needed
- **HTTP** transport for remote/Docker deployments - bearer-token auth, `/health`, and why it's a separate process from the client (unlike stdio)
- Wiring a Matimo MCP server into Claude Desktop's config

### Prerequisites
- Complete Notebook 01 (Quickstart) first
- No API key needed - this notebook only calls the built-in `calculator` tool, no LLM involved

> **Version note:** `matimo[mcp]` depends on `mcp>=1.28.1` with no upper bound, so `pip install` can resolve either the `mcp` 1.x or 2.x SDK - the two have an incompatible `Server` registration API. This notebook's `matimo-core` build detects the installed `mcp` major version and adapts automatically. If you hit `AttributeError: 'Server' object has no attribute 'list_tools'` on an older `matimo-core`, either upgrade `matimo-core` or pin `mcp<2.0`.

### Step 1 - Install Matimo with MCP support

In [ ]:
!pip install -q "matimo[mcp]"

### Step 2 - Write a skill and an MCP stdio server script

`MCPServer` wraps any `Matimo` instance and exposes its tools (and skills) over MCP. `tools=["calculator"]` allowlists a single tool to keep this demo tidy - omit `tools=` to expose everything the instance has loaded. `skill_paths` makes any skills in that directory available as MCP *resources*, which Step 4 reads.

The server has to live in its own file: stdio transport works by the *client* spawning the server as a subprocess and speaking JSON-RPC over its stdin/stdout, so there's no way to hand it an in-process object - this is exactly how Claude Desktop and Cursor launch a local MCP server.

In [1]:
import tempfile
from pathlib import Path

demo_dir = Path(tempfile.mkdtemp(prefix="matimo-mcp-demo-"))
skills_dir = demo_dir / "skills"
(skills_dir / "quickstart-tips").mkdir(parents=True)

(skills_dir / "quickstart-tips" / "SKILL.md").write_text("""---
name: quickstart-tips
description: Tips for getting started with Matimo quickly.
---

# Quickstart Tips

Initialize Matimo with `auto_discover=True` to pick up every installed provider package automatically.
""")

server_script = demo_dir / "mcp_stdio_server.py"
server_script.write_text(f'''
import asyncio

async def main():
    from matimo import Matimo
    from matimo.mcp.server import MCPServer, MCPServerOptions

    matimo = await Matimo.init(
        auto_discover=True,
        skill_paths=[{str(skills_dir)!r}],
        log_level="silent",
    )
    server = MCPServer(
        matimo,
        MCPServerOptions(transport="stdio", tools=["calculator"]),
    )
    await server.start()

if __name__ == "__main__":
    asyncio.run(main())
''')

print(f"Demo directory: {demo_dir}")
print(f"Skill written to: {skills_dir / 'quickstart-tips' / 'SKILL.md'}")
print(f"Server script written to: {server_script}")

Demo directory: /var/folders/1d/5sj004_10236f7xwyyjy5z5w0000gn/T/matimo-mcp-demo-ikrmu153
Skill written to: /var/folders/1d/5sj004_10236f7xwyyjy5z5w0000gn/T/matimo-mcp-demo-ikrmu153/skills/quickstart-tips/SKILL.md
Server script written to: /var/folders/1d/5sj004_10236f7xwyyjy5z5w0000gn/T/matimo-mcp-demo-ikrmu153/mcp_stdio_server.py


### Step 3 - Connect over stdio, discover and call a tool

`StdioServerParameters` + `stdio_client` spawns the script from Step 2 and manages its stdin/stdout as the JSON-RPC transport. `ClientSession.initialize()` performs the MCP handshake, `list_tools()` returns the (auth-stripped) tool schemas, and `call_tool()` runs one exactly like `matimo.execute()` would, just over the wire.

In [2]:
import sys

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

params = StdioServerParameters(command=sys.executable, args=[str(server_script)])

async with stdio_client(params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        tools_result = await session.list_tools()
        print(f"Discovered {len(tools_result.tools)} tool(s) over stdio:")
        for t in tools_result.tools:
            print(f"  - {t.name}: {t.description[:70]}")

        call_result = await session.call_tool("calculator", {"expression": "6 * 7"})
        print(f"\ncalculator(expression='6 * 7') -> isError={call_result.isError}")
        for block in call_result.content:
            if block.type == "text":
                print(block.text)

Discovered 1 tool(s) over stdio:
  - calculator: Perform arithmetic operations. Two mutually exclusive modes are suppor

calculator(expression='6 * 7') -> isError=False
{
  "result": 42.0,
  "operation": "expression",
  "original_operation": "expression",
  "operands": {
    "expression": "6 * 7"
  }
}


### Step 4 - Skills as MCP resources

Any skill under a `skill_paths` directory is automatically registered as an MCP resource at `skills://<name>` - a client with resource support (Claude Desktop, Cursor) can browse and read it directly, no `matimo_get_skill` tool call required. This is a fresh connection (stdio spawns a new server subprocess per client), reusing the same script from Step 2.

In [3]:
async with stdio_client(params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        resources_result = await session.list_resources()
        print(f"Discovered {len(resources_result.resources)} resource(s):")
        for r in resources_result.resources:
            print(f"  - {r.uri}  ({r.mimeType})")

        uri = resources_result.resources[0].uri
        read_result = await session.read_resource(uri)
        print(f"\nContent of {uri}:\n")
        for c in read_result.contents:
            if getattr(c, "text", None):
                print(c.text)

Discovered 1 resource(s):
  - skills://quickstart-tips  (text/markdown)

Content of skills://quickstart-tips:

# Quickstart Tips

Initialize Matimo with `auto_discover=True` to pick up every installed provider package automatically.


### Step 5 - HTTP transport (remote / Docker deployments)

Stdio only works when the client can spawn the server as a local subprocess. For a remote deployment, run the server as its own long-lived process and have clients connect over the network - Matimo's HTTP transport adds bearer-token auth, CORS, and a `/health` endpoint on top of MCP's Streamable HTTP.

Here we launch the server as a real background subprocess (not an in-process task) since that's the actual deployment shape - the notebook process and the server process are independent.

In [4]:
http_server_script = demo_dir / "mcp_http_server.py"
http_server_script.write_text('''
import asyncio
import os

async def main():
    from matimo import Matimo
    from matimo.mcp.server import MCPServer, MCPServerOptions

    matimo = await Matimo.init(auto_discover=True, log_level="silent")
    server = MCPServer(
        matimo,
        MCPServerOptions(
            transport="http",
            port=int(os.environ["MATIMO_MCP_PORT"]),
            tools=["calculator"],
            mcp_token=os.environ["MATIMO_MCP_TOKEN"],
        ),
    )
    await server.start()

if __name__ == "__main__":
    asyncio.run(main())
''')

print(f"HTTP server script written to: {http_server_script}")

HTTP server script written to: /var/folders/1d/5sj004_10236f7xwyyjy5z5w0000gn/T/matimo-mcp-demo-ikrmu153/mcp_http_server.py


In [5]:
import os
import subprocess
import time

import httpx

MCP_PORT = 3199
MCP_TOKEN = "demo-token"

env = {**os.environ, "MATIMO_MCP_PORT": str(MCP_PORT), "MATIMO_MCP_TOKEN": MCP_TOKEN}
http_proc = subprocess.Popen(
    [sys.executable, str(http_server_script)],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

for _ in range(30):
    try:
        if httpx.get(f"http://127.0.0.1:{MCP_PORT}/health", timeout=1.0).status_code == 200:
            break
    except Exception:
        pass
    time.sleep(0.5)
else:
    raise RuntimeError("MCP HTTP server never became healthy")

print(f"MCP HTTP server is healthy on port {MCP_PORT}")

MCP HTTP server is healthy on port 3199


Now connect with `streamablehttp_client`, passing the bearer token as an `Authorization` header - the same shape a Claude Desktop remote-MCP config or any HTTP MCP client would send.

In [6]:
from mcp.client.streamable_http import streamablehttp_client

async with streamablehttp_client(
    f"http://127.0.0.1:{MCP_PORT}/mcp",
    headers={"Authorization": f"Bearer {MCP_TOKEN}"},
) as (read, write, _):
    async with ClientSession(read, write) as session:
        await session.initialize()

        tools_result = await session.list_tools()
        print(f"Discovered {len(tools_result.tools)} tool(s) over HTTP")

        call_result = await session.call_tool("calculator", {"expression": "sqrt(144)"})
        print(f"calculator(expression='sqrt(144)') -> isError={call_result.isError}")
        for block in call_result.content:
            if block.type == "text":
                print(block.text)

Discovered 1 tool(s) over HTTP
calculator(expression='sqrt(144)') -> isError=False
{
  "result": 12.0,
  "operation": "expression",
  "original_operation": "expression",
  "operands": {
    "expression": "sqrt(144)"
  }
}


The bearer token isn't optional - a request without it is rejected before it reaches the MCP session:

In [7]:
response = httpx.post(f"http://127.0.0.1:{MCP_PORT}/mcp", json={}, timeout=2.0)
print(f"Request without a bearer token -> HTTP {response.status_code}")

Request without a bearer token -> HTTP 401


In [8]:
http_proc.terminate()
http_proc.wait(timeout=5)
print("MCP HTTP server stopped.")

MCP HTTP server stopped.


### Step 6 - Wiring into Claude Desktop or Cursor

The stdio script from Step 2 is already a valid local MCP server - point a client's config at it directly:

```json
{
  "mcpServers": {
    "matimo": {
      "command": "python",
      "args": ["/path/to/mcp_stdio_server.py"],
      "env": { "SLACK_BOT_TOKEN": "xoxb-..." }
    }
  }
}
```

Drop that in `claude_desktop_config.json` (Claude Desktop) or the equivalent MCP config for Cursor/Windsurf, restart the client, and every tool this notebook called programmatically shows up in the client's tool picker - governed by the same policy engine either way.

---
## Summary

| Concept | API | Notes |
|---|---|---|
| Stdio server | `MCPServer(matimo, MCPServerOptions(transport="stdio"))` | Client spawns the server as a subprocess; the Claude Desktop / Cursor integration path |
| HTTP server | `MCPServer(matimo, MCPServerOptions(transport="http", port=..., mcp_token=...))` | Runs as its own process; bearer-token auth, CORS, `/health` |
| Stdio client | `StdioServerParameters` + `mcp.client.stdio.stdio_client` | Official `mcp` Python SDK |
| HTTP client | `mcp.client.streamable_http.streamablehttp_client` | Pass the bearer token via `Authorization` header |
| Skills as resources | `skill_paths=[...]` → `skills://<name>` | `session.list_resources()` / `session.read_resource()` |
| Tool scoping | `MCPServerOptions(tools=[...])` / `exclude_tools=[...]` | Allowlist/denylist by name or glob |

### What's Next?
- `00_index.ipynb` - full notebook index and recommended learning path
- `04_skills.ipynb` - the skills system these resources are built on
- `03_meta_tools.ipynb` - agents that create their own tools at runtime
- GitHub: [tallclub/matimo](https://github.com/tallclub/matimo)

**If this was useful, please star the repo: https://github.com/tallclub/matimo**